In [3]:
import pandas as pd

movies= pd.read_csv("D:\MINI PROJECT\DATASET\movies_final.csv")

print(movies.shape)
movies.head()


(23138, 15)


,movieId,tmdb_id,title,overview,genres,keywords,director,cast_top5,runtime,release_year,popularity,vote_average,vote_count,poster_path,combined_text
0,1,862.0,Toy Story,"led by woody, andy's toys live happily in his ...","family, comedy, animation, adventure","rescue, friendship, mission, jealousy, villain...",John Lasseter,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",81.0,1995.0,17.9054,8.000,19336.0,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,"led by woody, andy's toys live happily in his ..."
1,2,8844.0,Jumanji,when siblings judy and peter discover an encha...,"adventure, fantasy, family","giant insect, board game, disappearance, jungl...",Joe Johnston,"Robin Williams, Kirsten Dunst, Bradley Pierce,...",104.0,1995.0,2.6696,7.243,10985.0,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,when siblings judy and peter discover an encha...
2,3,15602.0,Grumpier Old Men,a family wedding reignites the ancient feud be...,"romance, comedy","fishing, sequel, old man, best friend, wedding...",Howard Deutch,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",101.0,1995.0,1.9051,6.500,410.0,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,a family wedding reignites the ancient feud be...
3,4,31357.0,Waiting to Exhale,"cheated on, mistreated and stepped on, the wom...","comedy, drama, romance","based on novel or book, single mother, divorce...",Forest Whitaker,"Whitney Houston, Angela Bassett, Loretta Devin...",127.0,1995.0,2.3907,6.281,180.0,/qJU6rfil5xLVb5HpJsmmfeSK254.jpg,"cheated on, mistreated and stepped on, the wom..."
4,5,11862.0,Father of the Bride Part II,just when george banks has recovered from his ...,"comedy, family","daughter, baby, parent child relationship, mid...",Charles Shyer,"Steve Martin, Diane Keaton, Martin Short, Kimb...",106.0,1995.0,2.5283,6.272,780.0,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,just when george banks has recovered from his ...


In [1]:
#Model used-SBERT
#Because convert sentence to vectors and capture semantic meaning

#Loading SBERT model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\MINI PROJECT\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#Generating movie embeddings based on genre & overview and other text data
texts = movies["combined_text"].tolist()

movie_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)


Batches:   0%|          | 0/724 [00:00<?, ?it/s]

In [5]:
#Implement cosine similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [6]:
def recommend_movies_content_based(
    liked_movie_ids,
    movies_df,
    embeddings,
    top_n=10
):
    # Map movieId → index
    id_to_index = {
        mid: idx for idx, mid in enumerate(movies_df["movieId"])
    }

    liked_indices = [
        id_to_index[mid] for mid in liked_movie_ids
        if mid in id_to_index
    ]

    # Average embedding of liked movies
    user_vector = np.mean(embeddings[liked_indices], axis=0)

    # Compute similarity with all movies
    similarity_scores = cosine_similarity(
        user_vector.reshape(1, -1),
        embeddings
    )[0]

    # Rank movies
    top_indices = similarity_scores.argsort()[::-1]

    # Exclude already liked movies
    recommended_indices = [
        idx for idx in top_indices
        if movies_df.iloc[idx]["movieId"] not in liked_movie_ids
    ][:top_n]

    return movies_df.iloc[recommended_indices][
        ["movieId", "title", "genres"]
    ]


In [7]:
# Example: user liked these movies
liked_movies = [1, 32, 296]  # Toy Story, Twelve Monkeys, Pulp Fiction

recommendations = recommend_movies_content_based(
    liked_movies,
    movies,
    movie_embeddings,
    top_n=10
)

recommendations


,movieId,title,genres
17168,97306,Seven Psychopaths,"comedy, crime"
16544,94112,Twelve,"thriller, drama, action, crime"
14367,82244,13,"drama, thriller"
19833,109742,Cheap Thrills,"comedy, crime, drama, thriller"
3636,4108,Five Corners,"drama, crime, thriller"
15118,86892,The Man from Nowhere,"action, thriller, crime"
10026,47146,Lady Killer,"comedy, crime"
2741,3114,Toy Story 2,"animation, comedy, family"
11132,58146,Witless Protection,"comedy, action, adventure, crime"
1545,1785,King of New York,"thriller, crime"
